In [2]:
# %% [markdown]
# # 12_prompt_ablation_experiment.ipynb
# Addresses Reviewer 1 Concern #4: isolates the individual
# contribution of v2 prompt components (few-shot examples, concept
# descriptions) to accuracy, via ablation rather than asserting their
# value without evidence.
#
# THREE CONDITIONS, all built from the SAME underlying schema/compiler
# (financial_semantic_layer.yaml, semantic_compiler.py) with no
# changes to utils.py's core build_v2_prompt():
#
#   FULL      : the exact v2 prompt as used throughout Sections 4-5
#   NO_SHOT   : identical to FULL, few-shot examples block removed
#   NO_DESC   : identical to FULL, per-concept natural-language
#               descriptions stripped from the schema block (concept
#               names/types/value-map hints retained)
#
# PROTOCOL NOTE (per R1's request that prompt variants be frozen and
# not iteratively tuned against results): the three prompt-builder
# functions below are written ONCE, before any condition is run, and
# are not modified based on results. Any future change must be a new,
# clearly-labeled variant, not a silent edit to one of these three.
#
# SCOPE: runs on a stratified 20-question subset (not the full 60) to
# control API cost -- 5 iterations x 3 conditions = 300 calls per
# condition, 900 total. Disclosed as a smaller sample than the main
# experiments; findings here are suggestive of component
# contribution, not as precise as the main 60-question results.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import re
import random
import pandas as pd
import time
from utils import (
    _v2_compiler, sanitize_evidence, FEW_SHOT_EXAMPLES_V2,
    evaluate, DB_PATH, client, MODEL,
)

print(f"DB_PATH in use: {DB_PATH}")
print(f"Model: {MODEL}")

# %% [markdown]
# ## 0. Define the three FROZEN prompt-builder variants

# %%
def build_prompt_FULL(question: str, evidence: str = "") -> str:
    """Identical to utils.py's build_v2_prompt() -- the baseline condition."""
    evidence = sanitize_evidence(evidence)
    schema_block = _v2_compiler.build_concept_only_prompt_schema()
    join_block = _v2_compiler.build_join_hint_block()
    excluded_block = _v2_compiler.excluded_concepts_note()

    return f"""You are an expert SQL query generator using a Semantic Layer.
You do NOT have access to the physical database schema. You may ONLY
reference the business concepts listed below. A separate compiler
(which you cannot see) will translate your concept-SQL into real SQL
after you respond.

{schema_block}

{join_block}

{excluded_block}

## Critical Rules
1. Reference every column as `<cube>.<concept_name>` using ONLY the
   concept names listed above. NEVER invent or guess a physical
   column name.
2. Write joins as: JOIN <cube_name> ON <cube_a>.<cube_b>_link
3. Table/cube names: account, client, disp, loan, trans, card, district
4. Measures are already full aggregate expressions. Use them directly;
   do NOT wrap them in your own SUM(), COUNT(), AVG(), or CAST().
   Measures are only valid in SELECT or HAVING -- never in WHERE.
5. SQL clause order: SELECT → FROM → JOIN → WHERE → GROUP BY → HAVING
   → ORDER BY → LIMIT.
6. "eligible for loans" = has a record in the loan table. Do NOT filter
   by loan.status.
7. Use standard SQLite functions only (IFNULL not ISNULL, no NUMBER()).
8. Return ONLY your concept-SQL query. No explanation, no markdown, no
   backticks.
{FEW_SHOT_EXAMPLES_V2}

## Question
{question}

## Additional Evidence
{evidence if evidence else 'None'}
"""


def build_prompt_NO_SHOT(question: str, evidence: str = "") -> str:
    """FULL, with the few-shot examples block removed entirely."""
    evidence = sanitize_evidence(evidence)
    schema_block = _v2_compiler.build_concept_only_prompt_schema()
    join_block = _v2_compiler.build_join_hint_block()
    excluded_block = _v2_compiler.excluded_concepts_note()

    return f"""You are an expert SQL query generator using a Semantic Layer.
You do NOT have access to the physical database schema. You may ONLY
reference the business concepts listed below. A separate compiler
(which you cannot see) will translate your concept-SQL into real SQL
after you respond.

{schema_block}

{join_block}

{excluded_block}

## Critical Rules
1. Reference every column as `<cube>.<concept_name>` using ONLY the
   concept names listed above. NEVER invent or guess a physical
   column name.
2. Write joins as: JOIN <cube_name> ON <cube_a>.<cube_b>_link
3. Table/cube names: account, client, disp, loan, trans, card, district
4. Measures are already full aggregate expressions. Use them directly;
   do NOT wrap them in your own SUM(), COUNT(), AVG(), or CAST().
   Measures are only valid in SELECT or HAVING -- never in WHERE.
5. SQL clause order: SELECT → FROM → JOIN → WHERE → GROUP BY → HAVING
   → ORDER BY → LIMIT.
6. "eligible for loans" = has a record in the loan table. Do NOT filter
   by loan.status.
7. Use standard SQLite functions only (IFNULL not ISNULL, no NUMBER()).
8. Return ONLY your concept-SQL query. No explanation, no markdown, no
   backticks.

## Question
{question}

## Additional Evidence
{evidence if evidence else 'None'}
"""
    # NOTE: no {FEW_SHOT_EXAMPLES_V2} block -- the only difference
    # from build_prompt_FULL above.


def _strip_descriptions(schema_block: str) -> str:
    """
    Removes the free-text description portion of each dimension/
    measure line in the schema block, while keeping the concept name,
    type, and "Valid values" hint lines for value-mapped dimensions
    intact -- only the natural-language description sentence is
    removed.
    """
    lines = schema_block.split('\n')
    stripped_lines = []
    for line in lines:
        # matches lines like "  - region (string): Region name."
        m = re.match(r'^(\s*-\s*\w+\s*\([^)]*\)(?:\s*\[primary key\])?)\s*:\s*.*$', line)
        if m:
            stripped_lines.append(m.group(1) + ':')
        else:
            stripped_lines.append(line)
    return '\n'.join(stripped_lines)


def build_prompt_NO_DESC(question: str, evidence: str = "") -> str:
    """FULL, with per-concept description sentences stripped from the schema block."""
    evidence = sanitize_evidence(evidence)
    schema_block = _strip_descriptions(_v2_compiler.build_concept_only_prompt_schema())
    join_block = _v2_compiler.build_join_hint_block()
    excluded_block = _v2_compiler.excluded_concepts_note()

    return f"""You are an expert SQL query generator using a Semantic Layer.
You do NOT have access to the physical database schema. You may ONLY
reference the business concepts listed below. A separate compiler
(which you cannot see) will translate your concept-SQL into real SQL
after you respond.

{schema_block}

{join_block}

{excluded_block}

## Critical Rules
1. Reference every column as `<cube>.<concept_name>` using ONLY the
   concept names listed above. NEVER invent or guess a physical
   column name.
2. Write joins as: JOIN <cube_name> ON <cube_a>.<cube_b>_link
3. Table/cube names: account, client, disp, loan, trans, card, district
4. Measures are already full aggregate expressions. Use them directly;
   do NOT wrap them in your own SUM(), COUNT(), AVG(), or CAST().
   Measures are only valid in SELECT or HAVING -- never in WHERE.
5. SQL clause order: SELECT → FROM → JOIN → WHERE → GROUP BY → HAVING
   → ORDER BY → LIMIT.
6. "eligible for loans" = has a record in the loan table. Do NOT filter
   by loan.status.
7. Use standard SQLite functions only (IFNULL not ISNULL, no NUMBER()).
8. Return ONLY your concept-SQL query. No explanation, no markdown, no
   backticks.
{FEW_SHOT_EXAMPLES_V2}

## Question
{question}

## Additional Evidence
{evidence if evidence else 'None'}
"""

PROMPT_BUILDERS = {
    'FULL': build_prompt_FULL,
    'NO_SHOT': build_prompt_NO_SHOT,
    'NO_DESC': build_prompt_NO_DESC,
}

# %% [markdown]
# ## 0b. Sanity check the three variants before spending API budget

# %%
sample_prompt_full = build_prompt_FULL("sample question", "")
sample_prompt_noshot = build_prompt_NO_SHOT("sample question", "")
sample_prompt_nodesc = build_prompt_NO_DESC("sample question", "")

print(f"FULL length:    {len(sample_prompt_full):,} chars")
print(f"NO_SHOT length: {len(sample_prompt_noshot):,} chars (should be shorter than FULL)")
print(f"NO_DESC length: {len(sample_prompt_nodesc):,} chars (should be shorter than FULL)")

assert len(sample_prompt_noshot) < len(sample_prompt_full), "NO_SHOT should be shorter!"
assert len(sample_prompt_nodesc) < len(sample_prompt_full), "NO_DESC should be shorter!"
assert FEW_SHOT_EXAMPLES_V2.strip()[:30] not in sample_prompt_noshot, \
    "Few-shot content still present in NO_SHOT variant!"
assert "Valid values for filtering" in sample_prompt_nodesc, \
    "NO_DESC accidentally stripped the value-map hint lines -- check _strip_descriptions()"
print("\n✓ All sanity checks passed. Variants are correctly differentiated.")

print("\n--- NO_DESC schema block sample (first 800 chars) ---")
schema_block_check = _strip_descriptions(_v2_compiler.build_concept_only_prompt_schema())
print(schema_block_check[:800])

# %% [markdown]
# ## 1. Stratified subset (20 questions) -- same seed as the
#      independent-validation notebook for consistency

# %%
with open('credit_rating_questions_all.json', 'r', encoding='utf-8') as f:
    all_questions = json.load(f)

random.seed(42)
by_category = {}
for q in all_questions:
    by_category.setdefault(q['category'], []).append(q)

subset = []
for cat, qs in sorted(by_category.items()):
    by_diff = {}
    for q in qs:
        by_diff.setdefault(q['difficulty'], []).append(q)
    for diff, items in sorted(by_diff.items()):
        subset.append(random.choice(items))

print(f"Ablation subset size: {len(subset)} questions")
for q in subset:
    print(f"  {q['question_id']} ({q['category']}, {q['difficulty']})")

N_ITER = 5
print(f"\nTotal API calls: {len(subset)} questions x {N_ITER} iterations x "
      f"{len(PROMPT_BUILDERS)} conditions = {len(subset) * N_ITER * len(PROMPT_BUILDERS)}")

# %% [markdown]
# ## 2. Run all three conditions

# %%
RESULTS_PATH = './results_ablation.csv'

try:
    df_saved = pd.read_csv(RESULTS_PATH)
    results = df_saved.to_dict('records')
    done_keys = set()
    for (cond, qid), grp in df_saved.groupby(['condition', 'question_id']):
        if grp['iteration'].nunique() >= N_ITER:
            done_keys.add((cond, qid))
    print(f"Resuming — completed (condition, question) pairs: {len(done_keys)}")
except FileNotFoundError:
    results = []
    done_keys = set()
    print("Starting fresh.")

print("\n" + "=" * 70)
print("RUNNING ABLATION EXPERIMENT")
print("=" * 70)

for condition, builder_fn in PROMPT_BUILDERS.items():
    print(f"\n--- Condition: {condition} ---")
    for i, row in enumerate(subset):
        q_id = row['question_id']
        question = row['question']
        evidence = row['evidence']
        gold_sql = row['SQL']

        if (condition, q_id) in done_keys:
            print(f"[{condition}][{i+1:2d}/{len(subset)}] {q_id} — skipped (done)")
            continue

        iter_correct = 0
        iter_compile_err = 0
        iter_exec_err = 0

        for iteration in range(N_ITER):
            try:
                prompt = builder_fn(question, evidence)
                t0 = time.time()
                api_res = client.chat.completions.create(
                    model=MODEL, max_tokens=512,
                    messages=[{"role": "user", "content": prompt}]
                )
                latency_ms = round((time.time() - t0) * 1000)
                concept_sql = api_res.choices[0].message.content.strip()
            except Exception as e:
                results.append({
                    'condition': condition, 'question_id': q_id, 'iteration': iteration,
                    'is_correct': False, 'compile_error': None,
                    'execution_error': f"API call failed: {e}",
                    'concept_sql': None, 'compiled_sql': None, 'gold_sql': gold_sql,
                    'latency_ms': None, 'prompt_chars': None,
                })
                iter_exec_err += 1
                continue

            compile_result = _v2_compiler.compile(concept_sql)

            if compile_result['error']:
                results.append({
                    'condition': condition, 'question_id': q_id, 'iteration': iteration,
                    'is_correct': False, 'compile_error': compile_result['error'],
                    'execution_error': None, 'concept_sql': concept_sql,
                    'compiled_sql': None, 'gold_sql': gold_sql,
                    'latency_ms': latency_ms, 'prompt_chars': len(prompt),
                })
                iter_compile_err += 1
                continue

            ev = evaluate(compile_result['sql'], gold_sql)
            correct = bool(ev['is_correct']) if ev['is_correct'] is not None else False
            if correct:
                iter_correct += 1
            if ev.get('error'):
                iter_exec_err += 1

            results.append({
                'condition': condition, 'question_id': q_id, 'iteration': iteration,
                'is_correct': correct, 'compile_error': None,
                'execution_error': ev.get('error'), 'concept_sql': concept_sql,
                'compiled_sql': compile_result['sql'], 'gold_sql': gold_sql,
                'latency_ms': latency_ms, 'prompt_chars': len(prompt),
            })

            time.sleep(0.3)

        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False, encoding='utf-8-sig')
        print(f"[{condition}][{i+1:2d}/{len(subset)}] {q_id} "
              f"correct={iter_correct}/{N_ITER} compile_err={iter_compile_err}/{N_ITER} "
              f"exec_err={iter_exec_err}/{N_ITER}")

print("\n" + "=" * 70)
print(f"Ablation complete. Saved: {RESULTS_PATH}")
print("=" * 70)

# %% [markdown]
# ## 3. Compare conditions

# %%
df_final = pd.read_csv(RESULTS_PATH)

print("Overall accuracy by condition:")
print(df_final.groupby('condition')['is_correct'].agg(['mean', 'count']))

print("\nFailure taxonomy by condition:")
for cond in df_final['condition'].unique():
    sub = df_final[df_final['condition'] == cond]
    total = len(sub)
    n_correct = sub['is_correct'].sum()
    n_compile = sub['compile_error'].notna().sum()
    n_exec = sub['execution_error'].notna().sum()
    n_wrong = total - n_correct - n_compile - n_exec
    print(f"  {cond:10s}: correct={n_correct/total:.1%}  "
          f"compile_err={n_compile/total:.1%}  exec_err={n_exec/total:.1%}  "
          f"wrong_result={n_wrong/total:.1%}")

# %% [markdown]
# ## 4. Question-level paired comparisons (FULL vs each ablated variant)

# %%
from scipy import stats
import numpy as np

def question_level_accuracy(df, condition):
    return df[df['condition'] == condition].groupby('question_id')['is_correct'].mean()

q_full = question_level_accuracy(df_final, 'FULL')

for cond in ['NO_SHOT', 'NO_DESC']:
    q_cond = question_level_accuracy(df_final, cond)
    common = sorted(set(q_full.index) & set(q_cond.index))
    if len(common) < 2:
        print(f"\n{cond}: insufficient overlapping questions for a paired test")
        continue
    full_vals = q_full.loc[common].values
    cond_vals = q_cond.loc[common].values
    diff = cond_vals - full_vals

    print(f"\n=== FULL vs {cond} (n={len(common)} questions) ===")
    print(f"FULL mean accuracy:  {full_vals.mean():.1%}")
    print(f"{cond} mean accuracy: {cond_vals.mean():.1%}")
    print(f"Mean diff ({cond} - FULL): {diff.mean():+.1%}")

    if len(common) >= 5:
        t_stat, t_p = stats.ttest_rel(cond_vals, full_vals)
        print(f"Paired t-test: t={t_stat:.3f}, p={t_p:.4g}")

print("""

INTERPRETATION:
- A large accuracy drop for NO_SHOT relative to FULL indicates the
  few-shot examples materially contribute to accuracy.
- A large accuracy drop for NO_DESC relative to FULL indicates the
  natural-language descriptions materially contribute.
- This ablation runs on a 20-question subset (not the full 60);
  treat findings as suggestive of component contribution, not as
  precise as the main experiments -- report the subset size
  explicitly alongside any ablation finding in the manuscript.
""")

SCRIPT VERSION: 2026-07-15-v1
DB_PATH in use: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
Model: gpt-4o-mini
FULL length:    13,967 chars
NO_SHOT length: 7,026 chars (should be shorter than FULL)
NO_DESC length: 12,180 chars (should be shorter than FULL)

✓ All sanity checks passed. Variants are correctly differentiated.

--- NO_DESC schema block sample (first 800 chars) ---
# Semantic Layer: Available Concepts

Below are business concepts you may use in your query. Concept names are NOT physical column names -- do not guess or invent a physical column name. Reference each concept as `<cube>.<concept_name>`.

## account
Customer account information. Central table linking loans, transactions, and orders.

Can be joined to:
  - district (many_to_one)

Dimensions:
  - account_id (number) [primary key]:
  - frequency (string):
  - open_date (time):

Measures (pre-defined aggregates -- use as-is, do not wrap in your own aggregate function):
  - count (count):

